# NB04 — Two-Scale Tiling & Token Budget

Builds tile-coordinate manifests at two target resolutions (0.5 and 2.0 μm/pixel). For each slide, candidate tiles are filtered by tissue coverage (≥30%), then uniformly sub-sampled to the per-scale token budget (1,200 high-res, 400 low-res). The pyramid level closest to each target mpp is selected per-slide from the slide metadata. Output is one parquet per slide per scale plus a tiling summary.

In [ ]:
import os, sys, math, json, time, random, datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
import openslide

WORKSPACE = Path(os.environ.get('WORKSPACE', './workspace'))
WSI_ROOT  = Path(os.environ.get('WSI_ROOT',  './data/wsi'))
SUBDIRS = {
    'compute':   WORKSPACE / 'compute',
    'logs':      WORKSPACE / 'logs',
    'figures':   WORKSPACE / 'figures',
    'qc':        WORKSPACE / 'qc',
    'tiles':     WORKSPACE / 'tiles',
    'manifests': WORKSPACE / 'manifests',
}
for p in SUBDIRS.values():
    p.mkdir(parents=True, exist_ok=True)

MANIFEST_PARQUET    = SUBDIRS['manifests'] / 'manifest_tcga.parquet'
QC_METRICS_PARQUET  = SUBDIRS['qc'] / 'qc_metrics_tcga.parquet'
TILES_DIR           = SUBDIRS['tiles'] / 'manifests'
TILES_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = SUBDIRS['figures']

SEED = 1337
random.seed(SEED); np.random.seed(SEED)

QC_POLICY = 'medium'
TARGET_SCALES = [0.5, 2.0]
TILE_SIZE = 256
OVERLAP   = 32
STRIDE    = TILE_SIZE - OVERLAP
MAX_TOKENS = {0.5: 1200, 2.0: 400}
MIN_TILE_TISSUE_COVERAGE = 0.30
MASK_MAX_SIDE = 2048
HSV_S_TISSUE_MIN = 20
HSV_V_WHITE_MIN  = 230
MAX_WORKERS = min(6, (os.cpu_count() or 8))
FORCE_REDO = False

def now_iso():
    return datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def choose_level_for_target_mpp(slide, target_mpp, fallback_base_mpp=0.25):
    props = slide.properties
    base_mpp = None
    for k in ('openslide.mpp-x', 'aperio.MPP'):
        if k in props:
            try:
                base_mpp = float(props.get(k)); break
            except Exception:
                pass
    if base_mpp is None:
        base_mpp = fallback_base_mpp
    best_level = 0; best_mpp = base_mpp
    for lvl in range(slide.level_count):
        mpp = base_mpp * slide.level_downsamples[lvl]
        if abs(mpp - target_mpp) < abs(best_mpp - target_mpp):
            best_mpp = mpp; best_level = lvl
    return best_level, float(best_mpp)

def make_tissue_mask(slide, max_side=MASK_MAX_SIDE):
    w, h = slide.dimensions
    scale = max(w, h) / max_side if max(w, h) > max_side else 1.0
    tw, th = int(w / scale), int(h / scale)
    thumb = slide.get_thumbnail((tw, th)).convert('RGB')
    a = np.array(thumb.convert('HSV'), dtype=np.uint8)
    H, S, V = a[..., 0], a[..., 1], a[..., 2]
    tissue = (S >= HSV_S_TISSUE_MIN) & (V < HSV_V_WHITE_MIN)
    return thumb, tissue

def grid_positions(level_w, level_h, tile=TILE_SIZE, stride=STRIDE):
    xs = list(range(0, max(level_w - tile, 0) + 1, stride))
    ys = list(range(0, max(level_h - tile, 0) + 1, stride))
    return xs, ys

def coverage_from_mask(mask, level, level_to_mask_scale, x, y, tile=TILE_SIZE):
    sx, sy = level_to_mask_scale
    mx0, my0 = int(x * sx), int(y * sy)
    mx1, my1 = int((x + tile) * sx), int((y + tile) * sy)
    mx0, my0 = max(mx0, 0), max(my0, 0)
    mx1, my1 = min(mx1, mask.shape[1]-1), min(my1, mask.shape[0]-1)
    if mx1 <= mx0 or my1 <= my0:
        return 0.0
    roi = mask[my0:my1, mx0:mx1]
    return float(roi.mean())

def sample_tiles_uniform(coords, k, rng):
    if len(coords) <= k:
        return coords
    idx = rng.choice(len(coords), size=k, replace=False)
    return [coords[i] for i in idx]

def write_parquet(df, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)

print(f'[{now_iso()}] loading manifest: {MANIFEST_PARQUET}')
df_manifest = pd.read_parquet(MANIFEST_PARQUET)

df_qc = None
if QC_POLICY in ('strict', 'medium') and QC_METRICS_PARQUET.exists():
    df_qc = pd.read_parquet(QC_METRICS_PARQUET)
    df_qc = df_qc[['slide_id', 'tissue_pct', 'white_pct', 'pen_pct', 'reasons']].copy()

if QC_POLICY == 'strict' and df_qc is not None:
    keep = df_manifest.merge(df_qc[['slide_id', 'reasons']], on='slide_id', how='left')
    keep = keep[keep['reasons'].isna() | (keep['reasons'] == '')]
elif QC_POLICY == 'medium' and df_qc is not None:
    keep = df_manifest.merge(df_qc, on='slide_id', how='left')
    keep = keep[(keep['tissue_pct'].fillna(1.0) >= 0.05) & (keep['white_pct'].fillna(0.0) <= 0.95)].copy()
else:
    keep = df_manifest.copy()
keep = keep.reset_index(drop=True)
print(f'[INFO] slides selected under QC policy {QC_POLICY!r}: {len(keep):,} of {len(df_manifest):,}')

def process_slide(row):
    slide_path = Path(row['path']); slide_id = str(row['slide_id'])
    cancer = str(row.get('cancer_code', 'UNKNOWN'))
    outputs = []; errors = []
    try:
        slide = openslide.OpenSlide(str(slide_path))
    except Exception as e:
        return slide_id, cancer, None, [f'OpenSlideError: {e}']
    try:
        thumb_rgb, mask = make_tissue_mask(slide, MASK_MAX_SIDE)
    except Exception as e:
        slide.close(); return slide_id, cancer, None, [f'MaskBuildError: {e}']
    level_dims = [slide.level_dimensions[i] for i in range(slide.level_count)]
    base_w, base_h = level_dims[0]

    for target in TARGET_SCALES:
        out_path = TILES_DIR / f"{slide_id}_scale{str(target).replace('.', 'p')}.parquet"
        if out_path.exists() and not FORCE_REDO:
            outputs.append({'scale': target, 'manifest': str(out_path), 'n_tiles': None, 'skipped': True})
            continue
        try:
            level, approx_mpp = choose_level_for_target_mpp(slide, target)
            level_w, level_h = level_dims[level]
            tw, th = thumb_rgb.size
            sx = tw / base_w; sy = th / base_h
            ds = slide.level_downsamples[level]
            level_to_mask_scale = (sx * ds, sy * ds)
            xs, ys = grid_positions(level_w, level_h, TILE_SIZE, STRIDE)
            cand = []
            for y in ys:
                for x in xs:
                    cov = coverage_from_mask(mask, level, level_to_mask_scale, x, y, TILE_SIZE)
                    if cov >= MIN_TILE_TISSUE_COVERAGE:
                        cand.append((x, y))
            budget = MAX_TOKENS.get(target, 0)
            rng = np.random.default_rng(SEED + (hash(slide_id) % (2**16)) + int(target * 100))
            chosen = sample_tiles_uniform(cand, budget, rng)
            data = []
            for idx, (x, y) in enumerate(chosen):
                data.append({
                    'slide_id': slide_id, 'cancer_code': cancer,
                    'scale_um_per_px': float(target),
                    'level': int(level), 'x': int(x), 'y': int(y),
                    'tile_size': TILE_SIZE, 'overlap': OVERLAP,
                    'approx_mpp': approx_mpp, 'tile_idx': int(idx),
                    'seed': int(SEED),
                })
            df_tiles = pd.DataFrame.from_records(data)
            write_parquet(df_tiles, out_path)
            outputs.append({'scale': target, 'manifest': str(out_path), 'n_tiles': len(df_tiles), 'skipped': False})
        except Exception as e:
            errors.append(f'TilingError(scale={target}): {e}')
    slide.close()
    return slide_id, cancer, outputs, errors

t0 = time.time(); done = 0
errors_all = []; per_slide_counts = []
print(f'[{now_iso()}] starting tiling on {len(keep)} slides with {MAX_WORKERS} workers')
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = [ex.submit(process_slide, row) for _, row in keep.iterrows()]
    for fut in as_completed(futs):
        slide_id, cancer, outputs, errs = fut.result()
        done += 1
        if errs:
            for e in errs:
                errors_all.append({'slide_id': slide_id, 'error': e})
        if outputs:
            for rec in outputs:
                if rec is None or rec.get('skipped', False):
                    continue
                per_slide_counts.append({
                    'slide_id': slide_id, 'cancer_code': cancer,
                    'scale_um_per_px': rec['scale'], 'n_tiles': rec['n_tiles'],
                    'manifest': rec['manifest'],
                })
        if done % 25 == 0 or done == len(keep):
            rate = done / (time.time() - t0 + 1e-9)
            print(f'  tiling {done}/{len(keep)} ({rate:.2f} slides/s)')

elapsed = time.time() - t0
print(f'[OK] tiling finished in {elapsed/60:.1f} min')

df_sum = pd.DataFrame.from_records(per_slide_counts)
sum_path = SUBDIRS['tiles'] / 'tiling_summary_tcga.parquet'
df_sum.to_parquet(sum_path, index=False)

if errors_all:
    err_path = SUBDIRS['tiles'] / 'tiling_errors_tcga.csv'
    pd.DataFrame.from_records(errors_all).to_csv(err_path, index=False, encoding='utf-8-sig')

for scale in TARGET_SCALES:
    df_sc = df_sum[df_sum['scale_um_per_px'] == scale]
    if len(df_sc) == 0: continue
    plt.figure(figsize=(8,5))
    plt.hist(df_sc['n_tiles'].dropna().values, bins=40)
    plt.xlabel(f'Tokens per slide @ {scale} μm/px'); plt.ylabel('Slides')
    plt.title(f'Token distribution @ {scale} μm/px')
    plt.tight_layout()
    outp = FIG_DIR / f"tiling_tokens_dist_scale{str(scale).replace('.', 'p')}.png"
    plt.savefig(outp, dpi=200); plt.close()

print(f'\n[OK] tiling summary: {sum_path}')
print('NB04 complete. Next: NB05 (frozen-backbone feature extraction).')